# YOLOv3 DPU Benchmark
Measures inference speed and power on the KV260 **FPGA DPU (B512)**.

YOLOv3 detects multiple objects with bounding boxes. Input: 416x416.

**Run from Jupyter** (`http://192.168.68.60:9090/lab`)

In [ ]:
import os, time, numpy as np

POWER_SENSOR = "/sys/class/hwmon/hwmon2/power1_input"
N_WARMUP, N_BENCHMARK = 5, 100

NOTEBOOK_DIR = os.path.dirname(os.path.abspath("dpu_bench.ipynb"))
XMODEL = (os.path.join(NOTEBOOK_DIR, "models", "tf_yolov3_voc.xmodel")
          if os.path.exists(os.path.join(NOTEBOOK_DIR, "models", "tf_yolov3_voc.xmodel"))
          else "/root/jupyter_notebooks/pynq-dpu/tf_yolov3_voc.xmodel")

def read_power_mw():
    with open(POWER_SENSOR) as f:
        return int(f.read().strip()) / 1000

print(f"xmodel: {XMODEL}")
print(f"Idle power: {read_power_mw()/1000:.2f} W")

## Step 1 — Load DPU Overlay

In [ ]:
from pynq_dpu import DpuOverlay
overlay = DpuOverlay("dpu.bit")
print("DPU overlay loaded!")

## Step 2 — Load YOLOv3 Model
3 output tensors — one per detection scale (13x13, 26x26, 52x52).

In [ ]:
overlay.load_model(XMODEL)
dpu = overlay.runner
in_t, out_t = dpu.get_input_tensors(), dpu.get_output_tensors()
print(f"Input:   {in_t[0].dims}  (416x416)")
print(f"Outputs: {[list(t.dims) for t in out_t]}")
print(f"  13x13=large, 26x26=medium, 52x52=small objects")

## Step 3 — Benchmark (zeros input)

In [ ]:
in_d  = [np.zeros(t.dims, dtype=np.int8) for t in in_t]
out_d = [np.zeros(t.dims, dtype=np.int8) for t in out_t]

for _ in range(N_WARMUP):
    job = dpu.execute_async(in_d, out_d); dpu.wait(job)
print("Warmup done!")

power_samples, start = [], time.time()
for i in range(N_BENCHMARK):
    job = dpu.execute_async(in_d, out_d); dpu.wait(job)
    if i % 10 == 0: power_samples.append(read_power_mw())
elapsed = time.time() - start

avg_power_w = sum(power_samples) / len(power_samples) / 1000
fps = N_BENCHMARK / elapsed
print("=" * 40)
print("YOLOV3 DPU BENCHMARK RESULTS")
print("=" * 40)
print(f"FPS:      {fps:.1f}")
print(f"Latency:  {elapsed/N_BENCHMARK*1000:.1f} ms/frame")
print(f"Power:    {avg_power_w:.2f} W")
print(f"FPS/Watt: {fps/avg_power_w:.2f}")
print("=" * 40)

In [ ]:
# Release benchmark overlay
del overlay
print("Benchmark overlay released.")

## Bonus — What Does YOLOv3 Detect? (Real Image Test)
**Preprocessing**: `pixels / 255` — standard YOLO normalization (different from ResNet50/InceptionV1).

VOC dataset: 20 classes. Fish is NOT a VOC class — model picks closest match.

This section uses its own separate overlay — delete or modify independently of the benchmark above.

In [ ]:
import cv2

VOC_CLASSES = ['aeroplane','bicycle','bird','boat','bottle','bus','car','cat',
               'chair','cow','diningtable','dog','horse','motorbike','person',
               'pottedplant','sheep','sofa','train','tvmonitor']

img_path = next((p for p in [
    "/root/jupyter_notebooks/dpu_benchmark/test_image_416.jpg",
    "/home/ubuntu/test_image_416.jpg"] if os.path.exists(p)), None)
img_rgb = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)

# Separate overlay for real image inference
overlay2 = DpuOverlay("dpu.bit")
overlay2.load_model(XMODEL)
dpu2 = overlay2.runner
in_t2, out_t2 = dpu2.get_input_tensors(), dpu2.get_output_tensors()

# YOLO uses pixels/255 normalization (different from ResNet50/InceptionV1)
img_int8 = (img_rgb.astype(np.float32) / 255.0 * 128).clip(-128, 127).astype(np.int8)

in_d2  = [np.zeros(t.dims, dtype=np.int8) for t in in_t2]
out_d2 = [np.zeros(t.dims, dtype=np.int8) for t in out_t2]
in_d2[0][0] = img_int8
job = dpu2.execute_async(in_d2, out_d2)
dpu2.wait(job)

# Parse: each output is (H, W, 75) = 3 boxes x 25 values (x,y,w,h,conf,20 classes)
best_conf, best_class = 0.0, 0
for out in out_d2:
    arr = out[0].astype(np.float32).reshape(-1, 25)
    confs = arr[:, 4]
    max_conf = float(confs.max())
    if max_conf > best_conf:
        best_conf = max_conf
        best_class = int(arr[int(confs.argmax()), 5:].argmax())

print(f"Image: {img_path} (416x416)")
print(f"YOLOv3 highest confidence detection:")
print(f"  Class: {VOC_CLASSES[best_class]}  (score: {best_conf:.2f})")
print(f"\nNote: Fish not in VOC — model picks closest match from 20 classes.")
print(f"VOC classes: {', '.join(VOC_CLASSES)}")

del overlay2
print("\nBonus overlay released.")